In [21]:
# ==========================================
# ECLAT - COMPLETE WORKING CODE
# ==========================================

import pandas as pd

# 1. Load dataset
file_path = "/content/Market_Basket_Optimisation.csv"

df = pd.read_csv(file_path, header=None)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Convert CSV into transactions
transactions = []

for i in range(len(df)):

    transaction = set()

    for j in range(len(df.columns)):

        value = df.iloc[i, j]

        if pd.notna(value):

            value = str(value).strip()

            if value != "":
                transaction.add(value)

    if transaction:
        transactions.append(transaction)

print("Number of transactions:", len(transactions))

# ==========================================
# 3. ECLAT settings
# ==========================================

min_support = 0.01
min_count = max(1, int(min_support * len(transactions)))

print("Minimum support:", min_support)
print("Minimum support count:", min_count)

# ==========================================
# 4. Create vertical TID database
# ==========================================

item_tid = {}

for tid, transaction in enumerate(transactions):

    for item in transaction:

        if item not in item_tid:
            item_tid[item] = set()

        item_tid[item].add(tid)

# ==========================================
# 5. Get frequent 1-itemsets
# ==========================================

frequent_items = {}

for item, tids in item_tid.items():

    if len(tids) >= min_count:

        frequent_items[frozenset([item])] = tids

print("Frequent 1-itemsets:", len(frequent_items))

# ==========================================
# 6. ECLAT algorithm
# ==========================================

all_frequent = {}

def eclat(prefix, items):

    for i in range(len(items)):

        item = items[i][0]
        tids = items[i][1]

        new_itemset = prefix | frozenset([item])

        if len(tids) >= min_count:

            all_frequent[new_itemset] = tids

            next_items = []

            for j in range(i + 1, len(items)):

                next_item = items[j][0]
                next_tids = items[j][1]

                # TID intersection
                intersection = tids & next_tids

                if len(intersection) >= min_count:

                    next_items.append(
                        (next_item, intersection)
                    )

            if next_items:

                eclat(
                    new_itemset,
                    next_items
                )

# ==========================================
# 7. Prepare items
# ==========================================

items = []

for itemset, tids in frequent_items.items():

    item = next(iter(itemset))

    items.append(
        (item, tids)
    )

items.sort(key=lambda x: x[0])

# ==========================================
# 8. Run ECLAT
# ==========================================

print("\nRunning ECLAT...")

eclat(
    frozenset(),
    items
)

print("ECLAT completed!")

# ==========================================
# 9. Create result table
# ==========================================

results = []

for itemset, tids in all_frequent.items():

    support_count = len(tids)

    support = support_count / len(transactions)

    results.append({
        "Itemset": " & ".join(sorted(itemset)),
        "Support": round(support, 4),
        "Support Count": support_count,
        "Itemset Size": len(itemset)
    })

result_df = pd.DataFrame(results)

# ==========================================
# 10. Sort results
# ==========================================

result_df = result_df.sort_values(
    by="Support",
    ascending=False
).reset_index(drop=True)

# ==========================================
# 11. Display output
# ==========================================

print("\n====================================")
print("ECLAT FREQUENT ITEMSETS")
print("====================================")

display(result_df.head(20))

# ==========================================
# 12. 2-itemsets
# ==========================================

print("\n====================================")
print("TOP 2-ITEMSETS")
print("====================================")

display(
    result_df[
        result_df["Itemset Size"] == 2
    ].head(20)
)

# ==========================================
# 13. 3-itemsets
# ==========================================

print("\n====================================")
print("TOP 3-ITEMSETS")
print("====================================")

display(
    result_df[
        result_df["Itemset Size"] == 3
    ].head(20)
)

Dataset loaded successfully
Rows: 7501
Columns: 20
Number of transactions: 7501
Minimum support: 0.01
Minimum support count: 75
Frequent 1-itemsets: 75

Running ECLAT...
ECLAT completed!

ECLAT FREQUENT ITEMSETS


,Itemset,Support,Support Count,Itemset Size
0,mineral water,0.2384,1788,1
1,eggs,0.1797,1348,1
2,spaghetti,0.1741,1306,1
3,french fries,0.1709,1282,1
4,chocolate,0.1638,1229,1
5,green tea,0.1321,991,1
6,milk,0.1296,972,1
7,ground beef,0.0983,737,1
8,frozen vegetables,0.0953,715,1
9,pancakes,0.0951,713,1



TOP 2-ITEMSETS


,Itemset,Support,Support Count,Itemset Size
21,mineral water & spaghetti,0.0597,448,2
23,chocolate & mineral water,0.0527,395,2
26,eggs & mineral water,0.0509,382,2
29,milk & mineral water,0.0480,360,2
34,ground beef & mineral water,0.0409,307,2
35,ground beef & spaghetti,0.0392,294,2
36,chocolate & spaghetti,0.0392,294,2
37,eggs & spaghetti,0.0365,274,2
38,eggs & french fries,0.0364,273,2
39,frozen vegetables & mineral water,0.0357,268,2



TOP 3-ITEMSETS


,Itemset,Support,Support Count,Itemset Size
129,ground beef & mineral water & spaghetti,0.0171,128,3
143,chocolate & mineral water & spaghetti,0.0159,119,3
145,milk & mineral water & spaghetti,0.0157,118,3
165,eggs & mineral water & spaghetti,0.0143,107,3
172,chocolate & milk & mineral water,0.0140,105,3
180,chocolate & eggs & mineral water,0.0135,101,3
189,eggs & milk & mineral water,0.0131,98,3
200,frozen vegetables & mineral water & spaghetti,0.0120,90,3
216,mineral water & pancakes & spaghetti,0.0115,86,3
223,frozen vegetables & milk & mineral water,0.0111,83,3
